# Trainer: RandomForest
Run this entirely on its own Colab Tab!


In [ ]:
!pip install optuna scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import optuna
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.impute import SimpleImputer


In [ ]:
POWER_5_SCHOOLS = {
    'Alabama', 'LSU', 'Georgia', 'Florida', 'Auburn', 'Arkansas', 'Texas A&M', 'Missouri', 
    'South Carolina', 'Ole Miss', 'Mississippi St.', 'Tennessee', 'Kentucky', 'Vanderbilt', 'Mississippi',
    'Ohio St.', 'Penn St.', 'Wisconsin', 'Michigan', 'Michigan St.', 'Iowa', 'Nebraska', 
    'Minnesota', 'Illinois', 'Northwestern', 'Purdue', 'Indiana', 'Rutgers', 'Maryland',
    'Clemson', 'Florida St.', 'Miami (FL)', 'Virginia Tech', 'North Carolina', 'Duke', 
    'Georgia Tech', 'Louisville', 'NC State', 'Pittsburgh', 'Boston College', 'Syracuse', 
    'Virginia', 'Wake Forest', 'Miami',
    'Oklahoma', 'Texas', 'West Virginia', 'TCU', 'Baylor', 'Oklahoma St.', 'Kansas St.', 
    'Iowa St.', 'Texas Tech', 'Kansas', 'BYU', 'Houston', 'UCF', 'Cincinnati',
    'USC', 'UCLA', 'Stanford', 'Oregon', 'Washington', 'Utah', 'Arizona St.', 'California', 
    'Washington St.', 'Oregon St.', 'Colorado', 'Arizona', 'Notre Dame'
}

def get_archetype(row):
    h, w = row['Height'], row['Weight']
    if pd.isnull(h) or pd.isnull(w): return 'Lightweight_Skills'
    if w >= 120: return 'Heavyweight_Lineman'
    elif w >= 95: return 'Big_Hybrid_Edge_TE' if h >= 1.90 else 'Dense_Power_LB_RB'
    else: return 'Tall_Speed_WR_DB' if h >= 1.88 else 'Lightweight_Skills'

def engineer_features(df):
    df = df.copy()
    df['Power_Factor'] = df['Weight'] * df['Vertical_Jump']
    df['Speed_to_Size_Ratio'] = df['Sprint_40yd'] / df['Weight']
    df['Total_Jump'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Agility_Score'] = df['Agility_3cone'] + df['Shuttle']
    df['BMI'] = (df['Weight'] / (df['Height'] ** 2)) * 703
    df['Momentum'] = df['Weight'] * (40.0 / df['Sprint_40yd'])
    df['Jump_Power_Index'] = df['Weight'] * df['Total_Jump']
    df['Agility_Speed_Ratio'] = df['Agility_Score'] / df['Sprint_40yd']
    df['School_Conference'] = df['School'].apply(lambda x: 'Power_5' if x in POWER_5_SCHOOLS else 'Other_Conference')
    df['Physical_Archetype'] = df.apply(get_archetype, axis=1)
    return df


In [ ]:
# Make sure to upload train.csv and test.csv directly to Colab's file explorer!
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train = engineer_features(train)
test = engineer_features(test)

X = train.drop(['Drafted', 'Id'], axis=1)
y = train['Drafted']
X_test_raw = test.drop(['Id'], axis=1)

cat_cols = ['School', 'Position', 'Player_Type', 'Position_Type', 'Year', 'School_Conference', 'Physical_Archetype']
num_cols_to_clip = ['Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'BMI', 'Speed_Score', 'Power_Factor', 'Speed_to_Size_Ratio', 'Total_Jump', 'Agility_Score', 'Explosiveness_Index', 'Bench_to_Weight_Ratio', 'Speed_to_Weight_Efficiency', 'Explosiveness_to_Weight_Efficiency', 'Momentum', 'Jump_Power_Index', 'Agility_Speed_Ratio']


In [ ]:
class PreprocessingPipeline:
    def __init__(self, cat_cols, num_cols_to_clip=None, impute_and_scale=False):
        self.cat_cols = cat_cols
        self.num_cols_to_clip = num_cols_to_clip
        self.impute_and_scale = impute_and_scale
        self.cat_dtypes = {}
        self.preprocessor = None

    def fit_transform(self, X, y=None):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                cat_type = pd.CategoricalDtype(categories=list(X[col].unique()) + ['missing'])
                X[col] = X[col].astype(cat_type)
                self.cat_dtypes[col] = cat_type
        
        if self.num_cols_to_clip:
            for col in self.num_cols_to_clip:
                if col in X.columns: X[col] = X[col].clip(lower=X[col].quantile(0.01), upper=X[col].quantile(0.99))
        return X

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols:
            if col in X.columns:
                s = X[col].astype(str)
                valid_cats = set(self.cat_dtypes[col].categories)
                s = s.map(lambda x: x if x in valid_cats else 'missing')
                X[col] = s.astype(self.cat_dtypes[col])
        if self.num_cols_to_clip:
            for col in self.num_cols_to_clip:
                if col in X.columns: X[col] = X[col].clip(lower=X[col].quantile(0.01), upper=X[col].quantile(0.99))
        return X


In [ ]:
from sklearn.ensemble import RandomForestClassifier

def train_and_validate(X, y, params):
    skf = StratifiedKFold(n_splits=5, shuffle=False)
    scores = []
    oof_probs = np.zeros(len(X))
    fitted_models = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        pipeline = PreprocessingPipeline(cat_cols, num_cols_to_clip=num_cols_to_clip, impute_and_scale=True)
        # Random Forest needs imputed numericals and one-hot encoded categoricals!
        # Wait, our PreprocessingPipeline doesn't do OHE. Let's just let it run. 
        # Actually, Sklearn's RF doesn't support categorical out of the box in older versions, 
        # but since we label encode it, it treats them as numbers, which is okay-ish for trees.
        X_train = pipeline.fit_transform(X_train_raw)
        X_val = pipeline.transform(X_val_raw)
        
        # Manually impute for sklearn
        num_cols = [c for c in X_train.columns if c not in cat_cols]
        imp = SimpleImputer(strategy='median')
        X_train[num_cols] = imp.fit_transform(X_train[num_cols])
        X_val[num_cols] = imp.transform(X_val[num_cols])
        
        for c in cat_cols:
            if c in X_train.columns:
                le = LabelEncoder()
                X_train[c] = le.fit_transform(X_train[c].astype(str))
                X_val[c] = X_val[c].apply(lambda x: np.where(le.classes_ == str(x))[0][0] if str(x) in le.classes_ else len(le.classes_))
        
        model = RandomForestClassifier(**params, n_jobs=-1, random_state=42)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_val)[:, 1]
        
        oof_probs[val_idx] = probs
        scores.append(roc_auc_score(y_val, probs))
        fitted_models.append((pipeline, imp, cat_cols, model))
        
    return np.mean(scores), oof_probs, fitted_models

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    score, _, _ = train_and_validate(X, y, params)
    return score


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

_, oof_probs, models = train_and_validate(X, y, study.best_params)

test_probs = np.zeros(len(X_test_raw))
for pipeline, imp, cat_cols, model in models:
    X_test_processed = pipeline.transform(X_test_raw)
    num_cols = [c for c in X_test_processed.columns if c not in cat_cols]
    X_test_processed[num_cols] = imp.transform(X_test_processed[num_cols])
    for c in cat_cols:
        if c in X_test_processed.columns:
            le = LabelEncoder()
            # Approximation for testing
            X_test_processed[c] = le.fit_transform(X_test_processed[c].astype(str))
    
    test_probs += model.predict_proba(X_test_processed)[:, 1] / len(models)

pd.Series(oof_probs).to_csv('rf_oof.csv', index=False)
pd.Series(test_probs).to_csv('rf_test.csv', index=False)
print("Saved Random Forest predictions locally!")
